# Fine-tune PP-OCRv5 mobile rec trên crop biển số Việt Nam

**Mục tiêu:** nâng độ chính xác chuỗi **biển 2 dòng** (hiện ~0,60 trên tập đánh giá 2.801 mẫu)
bằng cách fine-tune bộ nhận dạng ký tự trên đúng phân phối mà hệ thống thật đưa vào OCR
(strip 2-dòng-ghép-ngang, cao 64 px).

### Notebook tự nhận biết hai chế độ chạy

| Chế độ | Khi nào | Dữ liệu lấy từ đâu | Tốc độ |
|---|---|---|---|
| **LOCAL** | kernel Python chạy trên chính máy có kho mã (VS Code chọn kernel local) | thẳng từ `datasets/processed/rec_finetune` | CPU: ~2 giờ/epoch |
| **REMOTE** | kernel là runtime Colab / máy chủ khác (VS Code chọn kernel Colab, hoặc mở trên colab.research.google.com) | cần đưa `rec_finetune.zip` lên — cell 3 hướng dẫn | T4: ~3 phút/epoch |

Cell 1 in ra chế độ đang chạy. **Mọi cell sau đó tự điều chỉnh theo**, không phải sửa tay.

> Máy phát triển của đồ án không có GPU CUDA, nên chế độ REMOTE trên Colab T4 nhanh hơn
> khoảng 40 lần. Đổi lại phải đưa dữ liệu (~40 MB) lên runtime một lần.


In [1]:
# 1) Nhan biet moi truong va xac dinh duong dan — CHAY CELL NAY TRUOC TIEN
import io, os, shutil, subprocess, sys, urllib.request
from pathlib import Path

def find_repo():
    """Tim goc kho ma bang moc CLAUDE.md; None neu kernel khong chay cung may."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'CLAUDE.md').exists() and (candidate / 'ai' / 'inference').is_dir():
            return candidate
    return None

ROOT = find_repo()
LOCAL = ROOT is not None

if LOCAL:
    WORK = ROOT / 'training-work'
    DATA = ROOT / 'datasets' / 'processed' / 'rec_finetune'
    EXPORT = ROOT / 'models' / 'rec_finetuned'
    PRETRAINED = ROOT / 'models' / 'pretrained' / 'en_PP-OCRv5_mobile_rec_pretrained.pdparams'
    VENV_PY = WORK / 'venv' / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
else:
    # Runtime tu xa (Colab): khong co kho ma, moi thu dung ngay trong runtime.
    WORK = Path('/content')
    DATA = WORK / 'data' / 'rec_finetune'
    EXPORT = WORK / 'rec_finetuned'
    PRETRAINED = WORK / 'pretrained' / 'en_PP-OCRv5_mobile_rec_pretrained.pdparams'
    VENV_PY = Path(sys.executable)   # cai thang vao moi truong cua runtime

PADDLEOCR = WORK / 'PaddleOCR'
OUTPUT = WORK / 'output' / 'rec_vn'
DRIVE_CKPT = Path('/content/drive/MyDrive/DATN/rec_vn_ckpt')  # checkpoint tren Drive, song sot khi runtime chet

LOGDIR = WORK / 'logs'


def run(args, cwd=None, log_name=None, tail_on_fail=40):
    """Chay lenh, STREAM log ve notebook VA ghi ra tep.

    Vi sao khong dung subprocess.run: no khong bat output, nen tien trinh con
    ghi thang vao stdout o TANG HE DIEU HANH cua kernel. Khi kernel o xa
    (Colab, VS Code noi tu xa) phan do KHONG chay ve o notebook — no roi vao
    log kernel ma nguoi dung khong bao gio thay.

    Da xay ra that: lan chay 28/07/2026 ket thuc voi ma thoat 1, nhung o
    notebook chi co dung dong lenh va con so 1, khong mot dong loi nao. Khong
    the biet vi sao hong, nen khong the sua.

    Ham nay doc tung dong cua tien trinh con roi in lai bang print(flush=True)
    — duong duy nhat chac chan chay ve notebook — va dong thoi ghi vao tep,
    de log con lai sau khi o notebook bi xoa hoac phien Colab dut giua chung.
    Khi lenh that bai no in han ra duong dan log va {tail_on_fail} dong cuoi,
    vi phan cuoi la cho chua nguyen nhan.

    Args:
        args: Lenh va tham so.
        cwd: Thu muc lam viec cua tien trinh con.
        log_name: Ten tep log, mac dinh lay theo ten script duoc goi.
        tail_on_fail: So dong cuoi in ra khi ma thoat khac 0.

    Returns:
        Ma thoat cua tien trinh con. 0 la thanh cong.
    """
    args = [str(a) for a in args]
    print('$', ' '.join(args), flush=True)

    if log_name is None:
        log_name = Path(args[1]).stem if len(args) > 1 else 'run'
    LOGDIR.mkdir(parents=True, exist_ok=True)
    log_path = LOGDIR / f'{log_name}.log'

    # PYTHONUNBUFFERED: khong co no, tien trinh con Python thay dau ra la ONG
    # chu khong phai terminal nen gom dem theo khoi 8 KB — log ve nho giot va
    # tien do huan luyen nhin nhu bi treo.
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}

    tail = []
    with io.open(log_path, 'w', encoding='utf-8', errors='replace') as log_file:
        proc = subprocess.Popen(
            args,
            cwd=str(cwd) if cwd else None,
            stdout=subprocess.PIPE,
            # Gop stderr vao stdout: PaddleOCR ghi log qua module logging (ra
            # stderr) nhung traceback cung ra stderr. Tach hai luong thi thu tu
            # dong bi dao, doc khong ra chuyen gi xay ra truoc chuyen gi.
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            errors='replace',
            env=env,
        )
        for line in proc.stdout:
            line = line.rstrip('\n')
            print(line, flush=True)
            log_file.write(line + '\n')
            tail.append(line)
            del tail[:-tail_on_fail]
        code = proc.wait()

    if code != 0:
        print(f'\n>> THAT BAI — ma thoat {code}', flush=True)
        print(f'>> Log day du: {log_path}', flush=True)
        print(f'>> {len(tail)} dong cuoi:', flush=True)
        for line in tail:
            print('   |', line, flush=True)
    return code

print('CHE DO      :', 'LOCAL — co kho ma tren may nay' if LOCAL else 'REMOTE — kernel khong thay kho ma')
print('Kho ma      :', ROOT if LOCAL else '(khong co — xem cell 3)')
print('Thu muc lam :', WORK)
print('Du lieu     :', DATA, '|', 'CO' if (DATA / 'train.txt').exists() else 'CHUA CO')
print('PaddleOCR   :', 'CO' if PADDLEOCR.exists() else 'CHUA CO')
print('Trong so goc:', 'CO' if PRETRAINED.exists() else 'CHUA CO')


CHE DO      : REMOTE — kernel khong thay kho ma
Kho ma      : (khong co — xem cell 3)
Thu muc lam : /content
Du lieu     : /content/data/rec_finetune | CHUA CO
PaddleOCR   : CHUA CO
Trong so goc: CHUA CO


In [2]:
# 2) Moi truong chay PaddlePaddle 3.3.1
#    LOCAL : tao venv rieng trong training-work/ (khong dung backend/.venv cua he thong that)
#    REMOTE: cai thang vao runtime; ban GPU neu runtime co GPU
#    MOC: in 'paddle 3.3.1 san sang | CUDA: True' (REMOTE/T4) hoac 'CUDA: False' (LOCAL CPU)
gpu_here = shutil.which('nvidia-smi') is not None

if LOCAL:
    if not VENV_PY.exists():
        WORK.mkdir(exist_ok=True)
        run([sys.executable, '-m', 'venv', WORK / 'venv'])
        run([VENV_PY, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
        run([VENV_PY, '-m', 'pip', 'install', '-q', 'paddlepaddle==3.3.1'])
else:
    try:
        import paddle  # noqa: F401
    except ImportError:
        if gpu_here:
            # Ban GPU 3.x CHI co tren index cua Paddle; thieu -i thi pip tim PyPI (dung o 2.6.2).
            run([VENV_PY, '-m', 'pip', 'install', '-q', '--timeout', '300', '--retries', '8',
                 'paddlepaddle-gpu==3.3.1',
                 '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'])
        else:
            run([VENV_PY, '-m', 'pip', 'install', '-q', 'paddlepaddle==3.3.1'])

out = subprocess.run([str(VENV_PY), '-c',
    'import paddle; print(paddle.__version__, paddle.device.is_compiled_with_cuda())'],
    capture_output=True, text=True)
ver, has_cuda = out.stdout.split() if out.returncode == 0 else ('?', 'False')
USE_GPU = has_cuda == 'True'
print(f'paddle {ver} san sang | CUDA: {USE_GPU}')
if not USE_GPU:
    print('  -> Chay CPU: ~2 gio/epoch. Xem bang chien luoc o cell 6.')


$ /usr/bin/python3 -m pip install -q --timeout 300 --retries 8 paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 GB 531.7 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 39.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 34.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 46.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 612.1 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/

In [3]:
# 2b) Gan Google Drive — CHI khi chay tren Colab.
#     Truoc day cell nay goi thang google.colab, nen chay o che do LOCAL la
#     vo ngay dong dau bang ModuleNotFoundError, du LOCAL khong can Drive.
if LOCAL:
    print('Che do LOCAL — bo qua Drive, du lieu lay tu', DATA)
else:
    from google.colab import drive

    drive.mount('/content/drive')
    !ls -la /content/drive/MyDrive/DATN/


Mounted at /content/drive
total 77788
-rw------- 1 root root 79653977 Aug  2 09:53 rec_finetune.zip


### Máy có GPU nhưng `CUDA: False`?

Gỡ bản CPU rồi cài bản GPU (cờ `-i` là **bắt buộc**, bản GPU 3.x không có trên PyPI):

```
# LOCAL:  training-work/venv/Scripts/pip …   |   REMOTE: pip …
pip uninstall -y paddlepaddle
pip install paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
```

rồi chạy lại cell 2.


In [4]:
# 3) Du lieu huan luyen
#    MOC (sau khi gop bien hiem 02/08/2026): 10.275 dong train.txt / 1.311 val.txt.
#    Tang so voi ban cu (6.672/571) do hai thay doi:
#      - gop 521 bien vang/xanh  -> ngu lieu 2.801 -> 3.322 dong
#      - phat them MANH VUN (nua tren / nua duoi cua bien 2 dong) lam mau rieng,
#        de model hoc doc dung thu ma buoc phat hien chu dua cho no luc chay that
if not (DATA / 'train.txt').exists():
    if LOCAL:
        print('Chua co dataset, dang sinh tu corpus nhan (vai phut)...')
        py = ROOT / 'backend' / '.venv' / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
        run([py if py.exists() else sys.executable,
             ROOT / 'scripts' / 'dataset' / 'build_rec_finetune_set.py'], cwd=ROOT)
    else:
        DATA.parent.mkdir(parents=True, exist_ok=True)
        zips = [Path('/content/rec_finetune.zip'),
                Path('/content/drive/MyDrive/DATN/rec_finetune.zip')]
        found = next((z for z in zips if z.exists()), None)
        if found:
            print('Giai nen', found)
            run(['unzip', '-q', found, '-d', DATA.parent])
        else:
            print('=' * 68)
            print('CAN DUA DU LIEU LEN RUNTIME — chon MOT trong hai cach:')
            print()
            print('  Cach 1 (nhanh nhat) — keo tha tep vao runtime:')
            print('     Tren may co kho ma, nen thu muc:')
            print('         datasets/processed/rec_finetune  ->  rec_finetune.zip  (~97 MB)')
            print('     Roi keo tha tep do vao muc Files cua Colab (hoac dung o VS Code:')
            print('     chuot phai thu muc /content -> Upload). Xong chay lai cell nay.')
            print()
            print('  Cach 2 — qua Google Drive:')
            print('     Tai rec_finetune.zip len MyDrive/DATN/ roi chay:')
            print('         from google.colab import drive; drive.mount("/content/drive")')
            print('     Xong chay lai cell nay.')
            print('=' * 68)

if (DATA / 'train.txt').exists():
    for name in ('train.txt', 'val.txt'):
        n = sum(1 for _ in (DATA / name).open(encoding='utf-8'))
        print(f'{name}: {n} dong')
    print('vi du  :', (DATA / 'train.txt').open(encoding='utf-8').readline().strip())
    print('charset:', len((DATA / 'dict36.txt').read_text(encoding='utf-8').split()), 'ky tu')


Giai nen /content/drive/MyDrive/DATN/rec_finetune.zip
$ unzip -q /content/drive/MyDrive/DATN/rec_finetune.zip -d /content/data
train.txt: 10275 dong
val.txt: 1311 dong
vi du  : images/train_02742_tiny.jpg	30A76661
charset: 36 ky tu


In [5]:
# 4) Ma nguon PaddleOCR (chua tools/train.py) — clone neu chua co
#    MOC: in duong dan config va 'Config OK'.
if not PADDLEOCR.exists():
    WORK.mkdir(parents=True, exist_ok=True)
    run(['git', 'clone', '--depth', '1',
         'https://github.com/PaddlePaddle/PaddleOCR.git', PADDLEOCR])
    run([VENV_PY, '-m', 'pip', 'install', '-q', '-r', PADDLEOCR / 'requirements.txt'])

CONFIG = PADDLEOCR / 'configs/rec/PP-OCRv5/multi_language/en_PP-OCRv5_mobile_rec.yaml'
if not CONFIG.exists():
    found = list(PADDLEOCR.glob('configs/rec/**/*en_PP-OCRv5_mobile*.y*ml'))
    print('Duong dan mac dinh doi, tim thay:', found)
    CONFIG = found[0]
print('CONFIG =', CONFIG, '| Config OK')


$ git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git /content/PaddleOCR
Cloning into '/content/PaddleOCR'...
Updating files:  84% (2110/2492)
Updating files:  85% (2119/2492)
Updating files:  86% (2144/2492)
Updating files:  87% (2169/2492)
Updating files:  88% (2193/2492)
Updating files:  89% (2218/2492)
Updating files:  90% (2243/2492)
Updating files:  91% (2268/2492)
Updating files:  92% (2293/2492)
Updating files:  93% (2318/2492)
Updating files:  94% (2343/2492)
Updating files:  95% (2368/2492)
Updating files:  96% (2393/2492)
Updating files:  97% (2418/2492)
Updating files:  98% (2443/2492)
Updating files:  99% (2468/2492)
Updating files: 100% (2492/2492)
Updating files: 100% (2492/2492), done.
$ /usr/bin/python3 -m pip install -q -r /content/PaddleOCR/requirements.txt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
# 5) Bo trong so goc en_PP-OCRv5_mobile_rec
#    LOCAL : uu tien models/pretrained/ trong kho ma; chi tai khi that su khong co.
#    REMOTE: tai ve runtime (khoang 70 MB).
#    MOC: file ~70 MB. Vai tram BYTE nghia la trang loi JSON, KHONG phai model.
URL = ('https://paddle-model-ecology.bj.bcebos.com/paddlex/'
       'official_pretrained_model/en_PP-OCRv5_mobile_rec_pretrained.pdparams')

def ok(p):
    return p.exists() and p.stat().st_size > 10_000_000

if not ok(PRETRAINED):
    PRETRAINED.parent.mkdir(parents=True, exist_ok=True)
    alt = (WORK / 'pretrained' / PRETRAINED.name) if LOCAL else None
    if alt is not None and ok(alt):
        print('Chep tu ban sao trong training-work/ (khong dung mang)')
        shutil.copy2(alt, PRETRAINED)
    else:
        print('Dang tai trong so goc...')
        urllib.request.urlretrieve(URL, PRETRAINED)

print(f'{PRETRAINED}: {PRETRAINED.stat().st_size / 1e6:.1f} MB',
      '(OK)' if ok(PRETRAINED) else '(SAI — xoa va chay lai cell nay)')


Dang tai trong so goc...
/content/pretrained/en_PP-OCRv5_mobile_rec_pretrained.pdparams: 70.1 MB (OK)


### Chiến lược theo phần cứng

| Phần cứng | Mỗi epoch | 30 epoch | Ghi chú |
|---|---|---|---|
| **T4 (Colab)** | ~3 phút | ~1,5 giờ | chạy một lèo |
| **CPU 20 luồng** | ~2 giờ | ~60 giờ | cân nhắc `EPOCHS = 12`, hoặc chạy nhiều đêm |

**Checkpoint lưu mỗi epoch** và cell 6 **tự nối tiếp** nếu tìm thấy checkpoint cũ — dừng giữa
chừng lúc nào cũng có model dùng được, chạy lại là đi tiếp chứ không làm lại từ đầu.

*(Phiên chạy trước trên CPU dừng ở epoch 5, `acc = 0,166`, đường cong đang lên đều.)*


In [7]:
# 0) CHAN DOAN — chay cell nay khi cell 6 tra ve mot con so ma khong co log.
#
#    Cell nay CO TINH doc lap: no dung ! magic cua Jupyter chu khong dung
#    run(). Ly do: neu ban vua thay file .ipynb nhung CHUA chay lai cell 1
#    thi kernel van giu ban run() cu trong bo nho, va loi van bi nuot y het.
#
#    Chay tu tren xuong, DUNG LAI o buoc dau tien in ra loi.

PADDLEOCR_DIR = '/content/PaddleOCR'   # doi neu chay LOCAL

print('=' * 64)
print('BUOC 1 — GPU: co that hay khong  <-- nguyen nhan hay gap nhat')
print('=' * 64)
# Hai cau hoi KHAC NHAU, va nham lan giua chung la cai bay:
#   * may co GPU khong          -> nvidia-smi
#   * paddle DA CAI co dung duoc GPU khong -> is_compiled_with_cuda()
# Ban CPU cua paddle van cai duoc tren may co GPU, va van chay — cho toi
# khi co thu hoi den device id thi no vo ra AttributeError kho hieu.
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>&1 | tail -3
!python -c "import paddle; from paddle.base import core; print('paddle', paddle.__version__); print('build co CUDA :', core.is_compiled_with_cuda())" 2>&1 | grep -v -i warn | tail -5
print()
print('Neu dong tren la False thi PHAI cai lai ban GPU — xem cell 2.')
print('Chay tiep voi ban CPU se hong o tools/program.py dong 971:')
print("  device = 'gpu:{}'.format(dist.ParallelEnv().dev_id) if use_gpu else 'cpu'")
print('  -> AttributeError: ParallelEnv object has no attribute _device_id')

print()
print('=' * 64)
print('BUOC 2 — cay ma PaddleOCR co import duoc khong')
print('=' * 64)
!cd {PADDLEOCR_DIR} && python -c "import ppocr; print('ppocr OK')" 2>&1 | tail -25

print()
print('=' * 64)
print('BUOC 3 — train.py in ra gi khi chay that')
print('=' * 64)
# BAT BUOC truyen use_gpu=false o day. Khong truyen thi no lay mac dinh
# 'use_gpu: true' trong YAML, va tren mot ban paddle CPU dieu do tu sinh ra
# dung cai AttributeError o buoc 1 — tuc phep chan doan tu tao ra loi ma no
# dang di tim, roi che mat loi that nam sau do.
!cd {PADDLEOCR_DIR} && python tools/train.py -c {CONFIG} -o Global.use_gpu=false 2>&1 | tail -40


BUOC 1 — GPU: co that hay khong  <-- nguyen nhan hay gap nhat
name, memory.total [MiB]
Tesla T4, 15360 MiB
paddle 3.3.1
build co CUDA : True

Neu dong tren la False thi PHAI cai lai ban GPU — xem cell 2.
Chay tiep voi ban CPU se hong o tools/program.py dong 971:
  device = 'gpu:{}'.format(dist.ParallelEnv().dev_id) if use_gpu else 'cpu'
  -> AttributeError: ParallelEnv object has no attribute _device_id

BUOC 2 — cay ma PaddleOCR co import duoc khong
ppocr OK

BUOC 3 — train.py in ra gi khi chay that
[2026/08/02 10:21:03] ppocr INFO:                 max_text_length : 25
[2026/08/02 10:21:03] ppocr INFO:                 prob : 0.5
[2026/08/02 10:21:03] ppocr INFO:             RecAug : None
[2026/08/02 10:21:03] ppocr INFO:             MultiLabelEncode : 
[2026/08/02 10:21:03] ppocr INFO:                 gtc_encode : NRTRLabelEncode
[2026/08/02 10:21:03] ppocr INFO:             KeepKeys : 
[2026/08/02 10:21:03] ppocr INFO:                 keep_keys : ['image', 'label_ctc', 'label_gtc', '

In [ ]:
# 6) HUAN LUYEN
#    MOC: sau moi 200 iter in 'cur metric, acc: ...' — con so nay phai TANG DAN.

# Hoi lai paddle NGAY BAY GIO thay vi tin USE_GPU tinh o cell 2. Giua hai cell
# co the da qua mot lan Restart runtime, hoac Colab da doi sang may khong GPU —
# va USE_GPU cu thi van con True trong bo nho kernel.
#
# Neu bo qua buoc nay: Global.use_gpu=true di vao tools/program.py dong 971,
# no goi dist.ParallelEnv().dev_id, ma _device_id CHI duoc gan khi paddle
# bien dich co CUDA. Ban CPU -> AttributeError khong noi mot chu nao ve GPU.
_probe = subprocess.run(
    [str(VENV_PY), '-c',
     'from paddle.base import core; print(core.is_compiled_with_cuda())'],
    capture_output=True, text=True)
CUDA_NOW = _probe.stdout.strip().endswith('True')

if USE_GPU and not CUDA_NOW:
    print('!! paddle dang cai la ban CPU, nhung USE_GPU dang True.')
    print('!! Chay tiep se hong o tools/program.py dong 971 (AttributeError).')
    print('!! Ha xuong CPU. Muon dung GPU thi chay lai cell 2 tren runtime CO GPU.')
USE_GPU = USE_GPU and CUDA_NOW
print('Huan luyen tren:', 'GPU' if USE_GPU else 'CPU (~2 gio/epoch)')

EPOCHS  = 30 if USE_GPU else 12    # CPU: 12 epoch da du thay xu huong (xem bang tren)
BATCH   = 32 if USE_GPU else 64     # T4 15GB OOM voi 128; 32 vua du
WORKERS = 2 if USE_GPU else 0      # Windows/CPU: 0 worker de khong treo qua dem

OUTPUT.mkdir(parents=True, exist_ok=True)
opts = [
    f'Global.use_gpu={str(USE_GPU).lower()}',
    f'Global.character_dict_path={(DATA / "dict36.txt").as_posix()}',
    'Global.use_space_char=false',
    'Global.max_text_length=10',
    f'Global.epoch_num={EPOCHS}',
    'Global.save_epoch_step=1',
    'Global.eval_batch_step=[0,200]',
    'Global.print_batch_step=20',
    f'Global.save_model_dir={OUTPUT.as_posix()}',
    'Optimizer.lr.learning_rate=0.0001',
    'Optimizer.lr.warmup_epoch=1',
    f'Train.dataset.data_dir={DATA.as_posix()}',
    f'Train.dataset.label_file_list=[{(DATA / "train.txt").as_posix()}]',
    f'Train.sampler.first_bs={BATCH}',
    f'Train.loader.batch_size_per_card={BATCH}',
    f'Train.loader.num_workers={WORKERS}',
    f'Eval.dataset.data_dir={DATA.as_posix()}',
    f'Eval.dataset.label_file_list=[{(DATA / "val.txt").as_posix()}]',
    f'Eval.loader.batch_size_per_card={BATCH}',
    f'Eval.loader.num_workers={WORKERS}',
]
# -- Khoi phuc checkpoint tu Drive (neu runtime truoc da luu) --
if not LOCAL and DRIVE_CKPT.exists() and not (OUTPUT / 'latest.pdparams').exists():
    print('>> Khoi phuc checkpoint tu Drive...')
    OUTPUT.mkdir(parents=True, exist_ok=True)
    for f in DRIVE_CKPT.iterdir():
        if f.is_file():
            shutil.copy2(f, OUTPUT / f.name)
    print('>> Da khoi phuc:', [f.name for f in OUTPUT.iterdir()])

if (OUTPUT / 'latest.pdparams').exists():
    print('>> Tiep tuc tu checkpoint cu:', (OUTPUT / 'latest.pdparams'))
    opts.insert(1, f'Global.checkpoints={(OUTPUT / "latest").as_posix()}')
else:
    print('>> Bat dau tu trong so goc')
    opts.insert(1, f'Global.pretrained_model={PRETRAINED.as_posix()[:-9]}')

# -- Background thread: sync checkpoint len Drive moi khi co file moi --
import threading, time

_sync_stop = threading.Event()

def _bg_sync_to_drive():
    """Chay song song voi training. Cu 60 giay kiem tra OUTPUT,
    neu thay .pdparams moi hon ban tren Drive thi copy len."""
    seen = {}  # ten file -> mtime da sync
    while not _sync_stop.is_set():
        _sync_stop.wait(60)
        if _sync_stop.is_set():
            break
        try:
            DRIVE_CKPT.mkdir(parents=True, exist_ok=True)
            for f in OUTPUT.iterdir():
                if f.is_file() and f.suffix in ('.pdparams', '.pdopt', '.states'):
                    mt = f.stat().st_mtime
                    if seen.get(f.name) != mt:
                        shutil.copy2(f, DRIVE_CKPT / f.name)
                        seen[f.name] = mt
                        print(f'  [sync] {f.name} -> Drive', flush=True)
        except Exception as e:
            print(f'  [sync] loi: {e}', flush=True)

if not LOCAL:
    _sync_thread = threading.Thread(target=_bg_sync_to_drive, daemon=True)
    _sync_thread.start()
    print('>> Background sync: moi 60s se copy checkpoint moi len Drive')

run([VENV_PY, 'tools/train.py', '-c', CONFIG, '-o', *opts], cwd=PADDLEOCR)

# -- Sync lan cuoi sau khi train xong --
_sync_stop.set()
if not LOCAL:
    DRIVE_CKPT.mkdir(parents=True, exist_ok=True)
    synced = []
    for f in OUTPUT.iterdir():
        if f.is_file() and f.suffix in ('.pdparams', '.pdopt', '.states'):
            shutil.copy2(f, DRIVE_CKPT / f.name)
            synced.append(f.name)
    print(f'>> Sync cuoi: {len(synced)} file len Drive: {DRIVE_CKPT}')


Huan luyen tren: GPU
>> Bat dau tu trong so goc
>> Background sync: moi 60s se copy checkpoint moi len Drive
$ /usr/bin/python3 tools/train.py -c /content/PaddleOCR/configs/rec/PP-OCRv5/multi_language/en_PP-OCRv5_mobile_rec.yaml -o Global.use_gpu=true Global.pretrained_model=/content/pretrained/en_PP-OCRv5_mobile_rec_pretrained Global.character_dict_path=/content/data/rec_finetune/dict36.txt Global.use_space_char=false Global.max_text_length=10 Global.epoch_num=30 Global.save_epoch_step=1 Global.eval_batch_step=[0,200] Global.print_batch_step=20 Global.save_model_dir=/content/output/rec_vn Optimizer.lr.learning_rate=0.0001 Optimizer.lr.warmup_epoch=1 Train.dataset.data_dir=/content/data/rec_finetune Train.dataset.label_file_list=[/content/data/rec_finetune/train.txt] Train.sampler.first_bs=32 Train.loader.batch_size_per_card=32 Train.loader.num_workers=2 Eval.dataset.data_dir=/content/data/rec_finetune Eval.dataset.label_file_list=[/content/data/rec_finetune/val.txt] Eval.loader.batch_

In [ ]:
# 6b) THEO DOI — chay cell nay o mot o KHAC trong khi cell 6 dang huan luyen.
#     Log day du nam trong tep, nen van doc duoc ca khi o cua cell 6 bi xoa
#     hoac phien Colab dut giua chung.
import subprocess

log_path = LOGDIR / 'train.log'
print('Log      :', log_path, '|', 'CO' if log_path.exists() else 'CHUA CO')
print('Checkpoint:', OUTPUT)
print()

if log_path.exists():
    lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
    print(f'--- {len(lines)} dong, 25 dong cuoi ---')
    for line in lines[-25:]:
        print(' ', line)
    # Chi so duy nhat dang tin de biet huan luyen CO tien trien:
    # 'acc' phai TANG DAN qua cac lan eval. Dung nhin loss.
    accs = [ln for ln in lines if 'cur metric' in ln or 'best metric' in ln]
    print()
    print(f'--- {len(accs)} moc do do chinh xac, 5 moc cuoi ---')
    for line in accs[-5:]:
        print(' ', line)
    if not accs:
        print('  (chua co moc nao — huan luyen chua vao duoc vong lap epoch)')
else:
    print('Chua co log. Cell 6 chua chay, hoac chay ban cu chua co stream.')

print()
for name in ('latest.pdparams', 'best_accuracy.pdparams'):
    print(f'{name:24}', 'CO' if (OUTPUT / name).exists() else 'chua co')

if USE_GPU:
    subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used',
                    '--format=csv'])


### Về cảnh báo `shape ... not matched` lúc bắt đầu

Log sẽ in vài dòng `WARNING: The shape of model params head.ctc_head.fc.weight
paddle.Size([120, 37]) not matched with loaded params ... paddle.Size([120, 438])`.

**Đây là chủ đích, không phải lỗi.** Model gốc có 438 lớp ký tự (tiếng Anh đầy đủ); đồ án dùng
**charset 36 ký tự** (`0-9A-Z`, quyết định Phase 1) nên hai lớp đầu ra được khởi tạo lại còn
backbone vẫn nạp nguyên. Thấy `load pretrain successful` ngay sau đó là đúng.


In [ ]:
# 7) Danh gia checkpoint tot nhat tren tap val (1.311 mau, khong augment)
#    LUU Y: tap val nay CO 632 mau manh vun. Do la co y — lan fine-tune truoc
#    that bai vi val chi co nguyen anh bien, tuc do mot che do he thong KHONG
#    dung, nen no khong nhin thay duoc cho hong. Xem docs/reports/31.
#    MOC: GHI LAI con so 'acc' — day la so de doi chieu voi baseline.
run([VENV_PY, 'tools/eval.py', '-c', CONFIG, '-o',
     f'Global.use_gpu={str(USE_GPU).lower()}',
     f'Global.checkpoints={(OUTPUT / "best_accuracy").as_posix()}',
     f'Global.character_dict_path={(DATA / "dict36.txt").as_posix()}',
     'Global.use_space_char=false', 'Global.max_text_length=10',
     f'Eval.dataset.data_dir={DATA.as_posix()}',
     f'Eval.dataset.label_file_list=[{(DATA / "val.txt").as_posix()}]',
     f'Eval.loader.batch_size_per_card={BATCH}',
     f'Eval.loader.num_workers={WORKERS}'], cwd=PADDLEOCR)


In [ ]:
# 8) Xuat inference model
#    LOCAL : xuat thang vao models/rec_finetuned/ — dung cho ALPR_OCR_REC_MODEL_DIR tro toi.
#    REMOTE: xuat ra /content/rec_finetuned roi nen lai de tai ve may.
run([VENV_PY, 'tools/export_model.py', '-c', CONFIG, '-o',
     f'Global.checkpoints={(OUTPUT / "best_accuracy").as_posix()}',
     f'Global.character_dict_path={(DATA / "dict36.txt").as_posix()}',
     'Global.use_space_char=false', 'Global.max_text_length=10',
     f'Global.save_inference_dir={EXPORT.as_posix()}'], cwd=PADDLEOCR)

print()
for f in sorted(EXPORT.glob('*')):
    print(f'  {f.name:<28} {f.stat().st_size / 1e6:8.2f} MB')

if not LOCAL:
    run(['zip', '-r', '-q', str(WORK / 'rec_finetuned.zip'), EXPORT.name], cwd=WORK)
    print('\nDa nen:', WORK / 'rec_finetuned.zip')
    print('Tai tep nay ve may, giai nen vao:  <kho ma>/models/rec_finetuned/')
    try:
        from google.colab import files
        files.download(str(WORK / 'rec_finetuned.zip'))
    except Exception as error:
        print('(Tai thu cong tu muc Files —', type(error).__name__, ')')


In [ ]:
# 8b) TAI MODEL VE MAY — chay sau cell 8
#     Cach 1: copy len Drive (an toan, khong mat khi runtime chet)
#     Cach 2: download truc tiep qua trinh duyet

zip_path = Path("/content/rec_finetuned.zip")

if not zip_path.exists():
    print("Chua co rec_finetuned.zip — chay cell 8 truoc!")
else:
    size_mb = zip_path.stat().st_size / 1e6
    print(f"rec_finetuned.zip: {size_mb:.1f} MB")

    # --- Copy len Google Drive ---
    drive_dst = Path("/content/drive/MyDrive/DATN/rec_finetuned.zip")
    try:
        drive_dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(zip_path, drive_dst)
        print(f"Da copy len Drive: {drive_dst}")
    except Exception as e:
        print(f"Khong copy duoc len Drive: {e}")

    # --- Download qua trinh duyet ---
    try:
        from google.colab import files
        print("Dang tai ve trinh duyet...")
        files.download(str(zip_path))
    except ImportError:
        print("Khong phai Colab — lay file tu Drive hoac thu cong.")
    except Exception as e:
        print(f"Download that bai: {e}")
        print(f"Lay thu cong tu Drive: {drive_dst}")


In [ ]:
# 9) CHI CHAY O CHE DO LOCAL — thu model moi canh model goc tren cung mot anh
#    MOC: hai dong ket qua; ground truth cua 2dong-1.png la 59K1-201.73
if not LOCAL:
    print('Bo qua: can kho ma tren cung may. Chay cell nay sau khi da dua model ve may.')
else:
    test_img = ROOT / 'demo' / 'images' / '2dong-1.png'
    snippet = f'''
import sys; sys.path.insert(0, r"{ROOT}")
import cv2
from ai.inference.config import InferenceConfig
from ai.inference.recognizer import PaddleOcrRecognizer
img = cv2.imread(r"{test_img}")
for label, kw in (("model goc      ", {{}}), ("model fine-tune", {{"ocr_rec_model_dir": r"{EXPORT}"}})):
    cfg = InferenceConfig(model_path=r"{ROOT / 'models' / 'best.pt'}", **kw)
    print(label, repr(PaddleOcrRecognizer(cfg).recognize(img).raw_text))
'''
    py = ROOT / 'backend' / '.venv' / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
    run([py if py.exists() else sys.executable, '-c', snippet], cwd=ROOT)


## Bật model mới cho cả hệ thống

Sau khi model nằm ở `models/rec_finetuned/` (chế độ REMOTE: giải nén `rec_finetuned.zip` vào đó),
bật bằng **một biến môi trường** — mọi dây nối đã có sẵn trong mã:

```
# Chạy trực tiếp:  set ALPR_OCR_REC_MODEL_DIR=models/rec_finetuned
# Docker (.env):   ALPR_OCR_REC_MODEL_DIR=/app/models/rec_finetuned
```

Đường dẫn sai sẽ **báo lỗi ngay lúc khởi động** thay vì âm thầm chạy model gốc.

### Đo lại trước khi tin — bắt buộc

Chi tiết ở `ai/training/README-rec-finetune.md`:

1. `ai/evaluation/ocr_accuracy.py` toàn tập, so với baseline trong `docs/reports/16-*`
2. Ba bộ hồi quy: 16 ảnh lõi (`demo/images/expected.json`), 13 ca rescue, 3 video demo
3. Ablation: chạy cả khi bật và khi tắt biến môi trường

**Chỉ tiêu đặt trước:** biển 2 dòng tăng **≥ 5 điểm**, biển 1 dòng **không giảm**. Không đạt thì
gỡ cờ, giữ model gốc, ghi kết quả âm vào báo cáo — một thí nghiệm thất bại có số liệu vẫn là
nội dung tốt cho mục hạn chế của luận văn.
